In [12]:
import pandas as pd

In [13]:
df = pd.read_csv("./true_labels_vs_prediction_gpt-4o-mini_standardEval.csv")

In [14]:
df.head()

,predictions,true_labels,is_vul
0,secure,Safe,False
1,secure,Safe,False
2,secure,CWE-787,True
3,secure,Safe,False
4,CWE-476,CWE-416,True


In [15]:
df.shape

(35386, 3)

In [16]:
df['is_vul'].value_counts()

is_vul
False    28371
True      7015
Name: count, dtype: int64

In [31]:
missing_rows = df[df["predictions"].isna()]
len(missing_rows)

0

In [27]:
rate_limit_rows = df[
    df["predictions"].str.contains("Rate limit|429", na=False)
]
len(rate_limit_rows)

0

In [28]:
rate_limit_rows

,predictions,true_labels,is_vul


In [32]:
invalid_format_rows = df[
    ~df["predictions"].str.match(r"^(Secure|secure|CWE-\d+)$", na=False)
]
len(invalid_format_rows)

396

In [33]:
invalid_format_rows

,predictions,true_labels,is_vul
267,CWE- allocations without initialization (CWE-242),Safe,False
372,CWE- Buffer Overflow,CWE-444,True
678,"121, 130, 134",Safe,False
725,CWE- buffer overflow,Safe,False
881,787,Safe,False
...,...,...,...
35084,CWE- freeing of memory not owned (CWE-401),Safe,False
35248,"120, 134, 126",CWE-125,True
35262,CWE- reallocating memory incorrectly (CWE-789),Safe,False
35323,CWE- allocation of resources without limits or...,Safe,False


#### add the missed 19723 row manually

In [34]:
ROW_ID = 19723

manual_row = pd.DataFrame([{
    "predictions": "secure",
    "true_labels": "Safe",
    "is_vul": False
}])


In [35]:
df_fixed = pd.concat([
    df.iloc[:ROW_ID],
    manual_row,
    df.iloc[ROW_ID:]
]).reset_index(drop=True)

In [37]:
df_fixed.iloc[19723]

predictions    secure
true_labels      Safe
is_vul          False
Name: 19723, dtype: object

In [36]:
df_fixed.shape

(35387, 3)

In [38]:
df_fixed.to_csv("./true_labels_vs_prediction_gpt-4o-mini_standardEval.csv", index=False)

In [39]:
from AIeval import ModelEval, PlotModelResults

In [41]:
ModelEval(df=df_fixed,modelType='generation',filePath='./gpt-40-mini_standardEval_binary.txt',targetMethod='binary')

Model type : generation
Method type : binary
The input DataFrame:
  predictions true_labels  is_vul
0      secure        Safe   False
1      secure        Safe   False
2      secure     CWE-787    True
3      secure        Safe   False
4     CWE-476     CWE-416    True
Saving results to ./gpt-40-mini_standardEval_binary.txt

--- Evaluation Results ---
Model Type: generation
Method Type: binary
True Positives: 2780
True Negatives: 21660
False Positives: 6712
False Negatives: 4235
Total Population for Metrics: 35387
Precision: 29.29%
Recall: 39.63%
F1 Score: 0.3368
Accuracy: 69.06%

Results also saved to ./gpt-40-mini_standardEval_binary.txt


In [43]:
ModelEval(df=df_fixed,modelType='generation',filePath='./gpt-40-mini_standardEval_multiclass.txt',targetMethod='multiclass')

Model type : generation
Method type : multiclass
The input DataFrame:
  predictions true_labels  is_vul
0      secure        Safe   False
1      secure        Safe   False
2      secure     CWE-787    True
3      secure        Safe   False
4     CWE-476     CWE-416    True
Saving results to ./gpt-40-mini_standardEval_multiclass.txt

--- Evaluation Results ---
Model Type: generation
Method Type: multiclass
True Positives: 21872
True Negatives: 12194991
False Positives: 13524
False Negatives: 13515
Total Population for Metrics: 12243902
Precision: 61.79%
Recall: 61.81%
F1 Score: 0.6180
Accuracy: 99.78%

Results also saved to ./gpt-40-mini_standardEval_multiclass.txt


In [44]:
PlotModelResults(modelName='gpt-40-mini_Chat_Binary',datasetName='MegaVuln_standard_evaluation',resultsPath='./gpt-40-mini_standardEval_binary.txt',outputPath='./')

Results plot saved to: .//gpt-40-mini_Chat_Binary_results_MegaVuln_standard_evaluation.png


In [45]:
PlotModelResults(modelName='gpt-40-mini_Chat_Multiclass',datasetName='MegaVuln_standard_evaluation',resultsPath='./gpt-40-mini_standardEval_multiclass.txt',outputPath='./')

Results plot saved to: .//gpt-40-mini_Chat_Multiclass_results_MegaVuln_standard_evaluation.png
